# Cuaderno 2 - Entrenamiento y guardado del modelo

**Objetivo:** entrenar un modelo de clasificacion, evaluarlo y **guardarlo** en `models/modelo.pkl` para que la API (`app/`) pueda cargarlo.

Este cuaderno cubre la primera mitad del ciclo de vida del modelo:

```text
Datos -> Analisis descriptivo -> Preparacion -> Entrenamiento -> Evaluacion -> Modelo .pkl
```

> El entrenamiento se hace **aqui, una sola vez**. La API nunca reentrena: solo carga el `.pkl` y ejecuta `predict()`.

Se usa el conjunto **Iris** de `scikit-learn` como base de datos de prueba.

## 1. Importaciones y constantes

No se necesitan imputadores porque el conjunto no tiene valores nulos (ver cuaderno 1).

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Ruta del modelo. Funciona tanto si el cuaderno se ejecuta desde notebooks/
# como si se ejecuta desde la raiz del proyecto.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = PROJECT_ROOT / "models" / "modelo.pkl"

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"

print(f"Raiz del proyecto : {PROJECT_ROOT}")
print(f"Modelo se guardara en: {MODEL_PATH}")

Raiz del proyecto : D:\GitHub\Modelo_guia_fastapi
Modelo se guardara en: D:\GitHub\Modelo_guia_fastapi\models\modelo.pkl


## 2. Preparacion de los datos

Se renombran las columnas a `snake_case`. **Estos nombres son el contrato con la API**: deben ser identicos a los campos del esquema Pydantic.

In [2]:
iris = load_iris(as_frame=True)

df = iris.frame.rename(
    columns={
        "sepal length (cm)": "sepal_length",
        "sepal width (cm)": "sepal_width",
        "petal length (cm)": "petal_length",
        "petal width (cm)": "petal_width",
    }
)
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

X = df[FEATURES]
y = df[TARGET]

print(f"X: {X.shape} | y: {y.shape}")
print(f"Clases: {sorted(y.unique())}")
X.head()

X: (150, 4) | y: (150,)
Clases: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## 3. Division entrenamiento / prueba

`random_state=42` fija la semilla para que el resultado sea **reproducible**.
`stratify=y` conserva la proporcion de clases en ambos subconjuntos.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba       : {X_test.shape[0]} registros")

Entrenamiento: 120 registros
Prueba       : 30 registros


## 4. El pipeline de scikit-learn

Se usa un **`Pipeline`** con dos pasos:

1. `StandardScaler`: estandariza cada variable (media 0, desviacion 1). Es *aprendido*: guarda la media y la desviacion de **entrenamiento**.
2. `LogisticRegression`: clasificador lineal e interpretable que ademas expone `predict_proba()`.

**Por que guardar el pipeline completo y no solo el clasificador**

El escalador aprende parametros (`mean_`, `scale_`) con los datos de entrenamiento. Si al predecir volvieramos a ajustarlo con los datos nuevos, cada solicitud se escalaria contra si misma y las predicciones serian inconsistentes. Al serializar el `Pipeline` completo, `predict()` reutiliza los objetos **ya ajustados** durante el entrenamiento: mismas transformaciones, mismo orden de columnas, mismo resultado.

In [4]:
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

pipeline.fit(X_train, y_train)
print("Entrenamiento finalizado.")

Entrenamiento finalizado.


## 5. Que aprendio el pipeline

Estos atributos son los que la API leera para validar la entrada:

- `feature_names_in_`: nombres y **orden** de las variables de entrada.
- `classes_`: clases que puede predecir y el orden de las columnas de `predict_proba()`.

In [5]:
print("Variables de entrada (orden):", list(pipeline.feature_names_in_))
print("Clases                      :", list(pipeline.classes_))
print("Soporta predict_proba       :", hasattr(pipeline, "predict_proba"))
print()
print("Medias aprendidas por el escalador:")
for nombre, media, escala in zip(
    pipeline.feature_names_in_,
    pipeline.named_steps["scaler"].mean_,
    pipeline.named_steps["scaler"].scale_,
):
    print(f"  {nombre:<14} media={media:.3f}  desviacion={escala:.3f}")

Variables de entrada (orden): ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
Clases                      : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Soporta predict_proba       : True

Medias aprendidas por el escalador:
  sepal_length   media=5.842  desviacion=0.837
  sepal_width    media=3.048  desviacion=0.447
  petal_length   media=3.770  desviacion=1.761
  petal_width    media=1.205  desviacion=0.759


## 6. Evaluacion del modelo

La evaluacion se hace **antes** del despliegue. Si el modelo no sirve, no tiene sentido publicarlo.

In [6]:
y_pred = pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.9333

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



In [7]:
clases = list(pipeline.classes_)
matriz = confusion_matrix(y_test, y_pred, labels=clases)

pd.DataFrame(matriz, index=[f"real: {c}" for c in clases], columns=[f"pred: {c}" for c in clases])

,pred: setosa,pred: versicolor,pred: virginica
real: setosa,10,0,0
real: versicolor,0,9,1
real: virginica,0,1,9


## 7. Ejemplo de inferencia (lo mismo que hara la API)

La API recibe un JSON, lo convierte en un `DataFrame` de una fila con estas mismas columnas y llama a `predict()` / `predict_proba()`.

In [8]:
registro = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}

entrada = pd.DataFrame([registro], columns=FEATURES)
etiqueta = pipeline.predict(entrada)[0]
probabilidades = pipeline.predict_proba(entrada)[0]

print(f"Registro      : {registro}")
print(f"Prediccion    : {etiqueta}")
print("Probabilidades:")
for clase, probabilidad in zip(pipeline.classes_, probabilidades):
    print(f"  {clase:<12} {probabilidad:.4f}")

Registro      : {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}
Prediccion    : setosa
Probabilidades:
  setosa       0.9808
  versicolor   0.0192
  virginica    0.0000


## 8. Guardar el modelo en `models/modelo.pkl`

Se serializa el **pipeline completo** con `joblib` (recomendado por scikit-learn para objetos con arreglos NumPy grandes).

> `joblib`/`pickle` pueden ejecutar codigo al deserializar: carga solo archivos que tu mismo generaste.

In [9]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_PATH)

print(f"Modelo guardado en: {MODEL_PATH}")
print(f"Tamano del archivo: {MODEL_PATH.stat().st_size / 1024:.1f} KB")

Modelo guardado en: D:\GitHub\Modelo_guia_fastapi\models\modelo.pkl
Tamano del archivo: 2.1 KB


## 9. Verificacion: cargar el archivo recien guardado

Si el archivo se guardo bien, debe producir **exactamente** las mismas predicciones. Esta prueba se hace aqui para no descubrir el problema en produccion.

In [10]:
modelo_cargado = joblib.load(MODEL_PATH)

predicciones_originales = pipeline.predict(X_test)
predicciones_cargadas = modelo_cargado.predict(X_test)

print("Variables del modelo cargado:", list(modelo_cargado.feature_names_in_))
print("Clases del modelo cargado   :", list(modelo_cargado.classes_))
print("Las predicciones coinciden   :", bool(np.array_equal(predicciones_originales, predicciones_cargadas)))

Variables del modelo cargado: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
Clases del modelo cargado   : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Las predicciones coinciden   : True


## 10. Registro de versiones (compatibilidad)

Un `.pkl` de scikit-learn puede **fallar al cargarse** si las versiones de las librerias cambian. Guarda esta salida junto al modelo y replica esas versiones en el servidor (`requirements.txt`).

In [11]:
print(f"python       : {sys.version.split()[0]}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"joblib       : {joblib.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")

python       : 3.11.14
scikit-learn : 1.9.1
joblib       : 1.6.0
pandas       : 3.0.6
numpy        : 2.4.6


## 11. Como se conecta este cuaderno con la API

```text
Cuaderno 2                                  API (FastAPI)
------------                                -------------
joblib.dump(pipeline, modelo.pkl)   -->     JoblibModelGateway.load()  (al arrancar)
pipeline.feature_names_in_          -->     Validacion de las variables
pipeline.classes_                   -->     Etiquetas de las probabilidades
pipeline.predict(df)                -->     PredictUseCase + POST /predict
```

**Siguiente paso (fuera del cuaderno):** desde la raiz del proyecto, ejecutar la API y probarla.

```bash
uvicorn app.main:app --reload
```

Abrir `http://127.0.0.1:8000/docs` y probar `POST /predict`.